# 04 — Mean-reversion sweep review

MR-Session 2 research artifact for `mean_reversion_bb_stoch`. The notebook consumes only the six matched-timeframe runs from `mr_b1..b3` and imports the pre-WF gate from production code.

## PRIOR — recorded before loading MR sweep results (2026-07-12)

1. The bare BB core is expected to bleed during persistent trends. The modal outcome is therefore **marginal or failed standalone edge**, not a clean solid pass. This would test the base layer, not falsify the full MMS methodology, whose claimed protection sits in deferred pyramiding and sequential sizing.
2. `mr_b2` should produce more trades than `mr_b1`. Within b2, `bb_only` is expected to be at least as common as `bb_stoch` in the top 10 because MMS uses Stochastic for add-ons rather than the base entry. A decisive `bb_stoch` win would support the Beta adaptation instead.
3. The 15m b3 space should pay the largest raw-to-post microstructure penalty. A good raw result that disappears post-cost is not usable edge.
4. Deep-band b1 may struggle to clear 100 trades; b2 and b3 are more likely to clear the sample-size condition. Top-1 without a nearby top-5 cluster is treated as luck.
5. BTC and ETH direction should broadly agree if the mechanism is structural, but exact best parameters need not transfer. Strong positive results on only one symbol count against promotion.
6. Regime robustness is the likely failure point: fewer than three positive calendar-year bins is plausible even for a full-history hard-pass candidate.
7. Expected hard-filter survivors: 0–4 of 180 runs. More than that would be a positive surprise; zero survivors would trigger the pyramiding/engine decision rather than an immediate methodology pivot.

All post-sweep interpretation below must be compared with this prior.

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from algo_bot.engine.walkforward import WF_ELIGIBILITY_THRESHOLDS
from algo_bot.metrics import sharpe

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INDEX_CSV = ROOT / "results" / "experiments" / "index.csv"
REVIEW_JSON = ROOT / "results" / "experiments" / "mr_sweep_review.json"
SPACES = ["mr_b1.yaml", "mr_b2.yaml", "mr_b3.yaml"]
EXPECTED = {
    ("mr_b1.yaml", "BTC/USDT", "1h"),
    ("mr_b1.yaml", "ETH/USDT", "1h"),
    ("mr_b2.yaml", "BTC/USDT", "1h"),
    ("mr_b2.yaml", "ETH/USDT", "1h"),
    ("mr_b3.yaml", "BTC/USDT", "15m"),
    ("mr_b3.yaml", "ETH/USDT", "15m"),
}
GROUP = ["space_file", "symbol", "timeframe"]
CORE_PARAMS = ["bb_window", "bb_num_std", "entry_mode", "sl_pct", "side"]
WATCHLIST_PARAMS = ["bb_num_std", "stoch_oversold", "stoch_overbought", "arm_expiry_bars", "require_reclaim"]

/home/janek/anaconda3/envs/algo_bot/lib/python3.11/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

## Scope and integrity

If the global append-only index contains older MR runs, retain the latest deterministic 30 rows for each expected group. Fail closed unless all six matched-TF groups are complete.

In [2]:
idx = pd.read_csv(INDEX_CSV)
mr = idx.loc[(idx["strategy"] == "mean_reversion_bb_stoch") & idx["space_file"].isin(SPACES)].copy()
mr["created_at"] = pd.to_datetime(mr["created_at"], utc=True)
mr = mr.sort_values("created_at").groupby(GROUP, group_keys=False).tail(30).reset_index(drop=True)

actual = set(map(tuple, mr[GROUP].drop_duplicates().itertuples(index=False, name=None)))
sizes = mr.groupby(GROUP).size()
assert actual == EXPECTED, f"Unexpected groups: missing={EXPECTED - actual}, extra={actual - EXPECTED}"
assert (sizes == 30).all(), f"Incomplete groups:\n{sizes}"
assert len(mr) == 180

params_df = pd.json_normalize(mr["params"].map(json.loads)).add_prefix("p:")
df = pd.concat([mr.reset_index(drop=True), params_df], axis=1)

# A log-return metric is undefined once equity reaches zero. The historical
# sweep predates the fail-closed guard added after this review, so audit every
# saved post-cost curve and retain the originally reported value separately.
equity_floors = {}
for run_id in df["run_id"]:
    equity_path = ROOT / "results" / "backtests" / run_id / "equity.csv"
    equity_floors[run_id] = pd.read_csv(equity_path, usecols=["Equity_adjusted"])["Equity_adjusted"].min()
df["post_equity_min"] = df["run_id"].map(equity_floors)
df["post_equity_valid"] = df["post_equity_min"] > 0
df["sharpe_post_reported"] = df["sharpe_post"]
df["sharpe_post_valid"] = df["sharpe_post"].where(df["post_equity_valid"])
print(sizes.to_string())
print(f"\nPost-cost bankruptcy: {(~df['post_equity_valid']).sum()}/{len(df)} runs; valid curves: {df['post_equity_valid'].sum()}")

space_file  symbol    timeframe
mr_b1.yaml  BTC/USDT  1h           30
            ETH/USDT  1h           30
mr_b2.yaml  BTC/USDT  1h           30
            ETH/USDT  1h           30
mr_b3.yaml  BTC/USDT  15m          30
            ETH/USDT  15m          30

Post-cost bankruptcy: 169/180 runs; valid curves: 11


## Heuristics A–D — clustering, distribution, cost spread, trade count

In [3]:
summary_rows = []
clustering_top10 = {}
top10_params = {}

for key, group in df.groupby(GROUP, sort=True):
    label = "|".join(key)
    ranked_reported = group.sort_values("sharpe_post_reported", ascending=False, na_position="last")
    ranked_valid = group.loc[group["post_equity_valid"]].sort_values("sharpe_post_valid", ascending=False)
    ranked_raw = group.sort_values("sharpe_raw", ascending=False, na_position="last")
    top10 = ranked_raw.head(10)  # valid fallback diagnostic when post-cost equity is insolvent
    valid_top5 = ranked_valid.head(5)["sharpe_post_valid"]
    top1 = ranked_valid["sharpe_post_valid"].iloc[0] if not ranked_valid.empty else np.nan
    top5_mean = valid_top5.mean()
    summary_rows.append({
        "space_file": key[0], "symbol": key[1], "timeframe": key[2],
        "n": len(group), "n_post_equity_valid": int(group["post_equity_valid"].sum()),
        "n_post_bankrupt": int((~group["post_equity_valid"]).sum()),
        "top1": top1, "top5_mean": top5_mean,
        "median": group["sharpe_post_valid"].median(),
        "pct_positive": (group["sharpe_post_valid"] > 0).sum() / len(group),
        "top1_top5_gap": top1 - top5_mean,
        "raw_top1": ranked_raw["sharpe_raw"].iloc[0],
        "raw_top5_mean": ranked_raw.head(5)["sharpe_raw"].mean(),
        "raw_median": group["sharpe_raw"].median(),
        "reported_post_top1_invalid_if_bankrupt": ranked_reported["sharpe_post_reported"].iloc[0],
        "raw_post_spread_mean": (group["sharpe_raw"] - group["sharpe_post_valid"]).mean(),
        "raw_post_spread_std": (group["sharpe_raw"] - group["sharpe_post_valid"]).std(),
        "top1_n_trades": ranked_valid["n_trades_post"].iloc[0] if not ranked_valid.empty else np.nan,
        "pct_n_trades_gt_100": (group["n_trades_post"] > 100).mean(),
    })
    clustering_top10[label] = {
        p: top10[f"p:{p}"].value_counts(dropna=False).to_dict()
        for p in dict.fromkeys(CORE_PARAMS + WATCHLIST_PARAMS) if f"p:{p}" in top10
    }
    top10_params[label] = [
        {
            "rank_raw": rank, "run_id": row.run_id,
            "sharpe_post_reported": row.sharpe_post_reported,
            "sharpe_post_valid": row.sharpe_post_valid,
            "post_equity_min": row.post_equity_min, "post_equity_valid": row.post_equity_valid,
            "sharpe_raw": row.sharpe_raw, "profit_factor_post": row.profit_factor_post,
            "max_drawdown_pct_post": row.max_drawdown_pct_post,
            "n_trades_post": row.n_trades_post, "params": json.loads(row.params),
        }
        for rank, row in enumerate(top10.itertuples(index=False), start=1)
    ]

summary = pd.DataFrame(summary_rows).sort_values(GROUP).reset_index(drop=True)
summary.round(3)

,space_file,symbol,timeframe,n,n_post_equity_valid,n_post_bankrupt,top1,top5_mean,median,pct_positive,top1_top5_gap,raw_top1,raw_top5_mean,raw_median,reported_post_top1_invalid_if_bankrupt,raw_post_spread_mean,raw_post_spread_std,top1_n_trades,pct_n_trades_gt_100
0,mr_b1.yaml,BTC/USDT,1h,30,6,24,-0.497,-0.832,-0.887,0.0,0.335,-0.291,-0.380,-1.084,0.176,0.223,0.180,2391.0,1.0
1,mr_b1.yaml,ETH/USDT,1h,30,0,30,NaN,NaN,NaN,0.0,NaN,-0.783,-0.848,-1.237,0.214,NaN,NaN,NaN,1.0
2,mr_b2.yaml,BTC/USDT,1h,30,5,25,-1.111,-1.335,-1.271,0.0,0.224,-1.062,-1.130,-1.386,-0.215,-0.002,0.056,1855.0,1.0
3,mr_b2.yaml,ETH/USDT,1h,30,0,30,NaN,NaN,NaN,0.0,NaN,-1.394,-1.630,-1.993,0.067,NaN,NaN,NaN,1.0
4,mr_b3.yaml,BTC/USDT,15m,30,0,30,NaN,NaN,NaN,0.0,NaN,-1.399,-1.518,-2.040,0.022,NaN,NaN,NaN,1.0
5,mr_b3.yaml,ETH/USDT,15m,30,0,30,NaN,NaN,NaN,0.0,NaN,-1.798,-1.860,-2.251,0.255,NaN,NaN,NaN,1.0


## Heuristic E — cross-symbol consistency

For each space, compare the sets of values represented by the top three BTC and ETH runs. This is a neighborhood-transfer diagnostic, not a demand for identical parameter vectors.

In [4]:
cross_symbol = {}
for space, group in df.groupby("space_file", sort=True):
    # All post-cost winners are bankrupt or deeply negative; use the valid raw
    # metric for the cross-symbol parameter-neighborhood diagnostic.
    by_symbol = {
        symbol: sg.sort_values("sharpe_raw", ascending=False).head(3)
        for symbol, sg in group.groupby("symbol")
    }
    overlap = {}
    for p in CORE_PARAMS:
        col = f"p:{p}"
        if col not in group:
            continue
        btc = set(by_symbol["BTC/USDT"][col].dropna().astype(str))
        eth = set(by_symbol["ETH/USDT"][col].dropna().astype(str))
        union = btc | eth
        overlap[p] = {"btc": sorted(btc), "eth": sorted(eth), "jaccard": len(btc & eth) / len(union) if union else None}
    symbol_top5 = summary.loc[summary["space_file"] == space].set_index("symbol")["raw_top5_mean"].to_dict()
    cross_symbol[space] = {
        "ranking_metric": "sharpe_raw (post-cost Sharpe invalid after bankruptcy)",
        "raw_top5_mean_by_symbol": symbol_top5,
        "both_positive": all(v > 0 for v in symbol_top5.values()),
        "parameter_overlap": overlap,
        "mean_core_jaccard": np.nanmean([v["jaccard"] for v in overlap.values()]),
    }
pd.DataFrame({k: {"both_positive": v["both_positive"], "mean_core_jaccard": v["mean_core_jaccard"]} for k, v in cross_symbol.items()}).T

,both_positive,mean_core_jaccard
mr_b1.yaml,False,0.766667
mr_b2.yaml,False,0.716667
mr_b3.yaml,False,0.833333


## Hard pre-WF filter — imported from ADR-013

In [5]:
t = WF_ELIGIBILITY_THRESHOLDS
eligible_mask = (
    df["post_equity_valid"]
    & (df["sharpe_post_valid"] > t["sharpe"])
    & (df["profit_factor_post"] > t["profit_factor"])
    & (df["n_trades_post"] > t["n_trades"])
    & (df["max_drawdown_pct_post"] > t["max_drawdown_pct"])
)
eligible = df.loc[eligible_mask].copy()
eligible["selection_score"] = eligible["sharpe_post_valid"] * eligible["n_trades_post"] / 1000
eligible = eligible.sort_values("selection_score", ascending=False)
print("WF_ELIGIBILITY_THRESHOLDS:", WF_ELIGIBILITY_THRESHOLDS)
eligible[[*GROUP, "run_id", "sharpe_post_valid", "profit_factor_post", "max_drawdown_pct_post", "n_trades_post", "selection_score"]]

WF_ELIGIBILITY_THRESHOLDS: {'sharpe': 1.0, 'profit_factor': 1.3, 'n_trades': 100.0, 'max_drawdown_pct': -0.2}


,space_file,symbol,timeframe,run_id,sharpe_post_valid,profit_factor_post,max_drawdown_pct_post,n_trades_post,selection_score


## Heuristic F — b2 entry-mode split

The top-10 count is the primary empirical MMS-vs-Beta resolution. Distribution statistics are included so the count cannot hide a lopsided random sample.

In [6]:
entry_mode_split = {}
b2 = df.loc[df["space_file"] == "mr_b2.yaml"]
for symbol, group in b2.groupby("symbol", sort=True):
    top10 = group.sort_values("sharpe_raw", ascending=False).head(10)
    counts = top10["p:entry_mode"].value_counts().reindex(["bb_only", "bb_stoch"], fill_value=0)
    stats = group.groupby("p:entry_mode")["sharpe_raw"].agg(["count", "max", "mean", "median"])
    entry_mode_split[symbol] = {"ranking_metric": "sharpe_raw", "top10_counts": counts.to_dict(), "distribution": stats.to_dict(orient="index")}
    print(f"{symbol} top-10:\n{counts.to_string()}\n\nAll samples:\n{stats.round(3).to_string()}\n")

BTC/USDT top-10:
p:entry_mode
bb_only     6
bb_stoch    4

All samples:
              count    max   mean  median
p:entry_mode                             
bb_only           9 -1.062 -1.259  -1.254
bb_stoch         21 -1.112 -1.444  -1.473

ETH/USDT top-10:
p:entry_mode
bb_only     3
bb_stoch    7

All samples:
              count    max   mean  median
p:entry_mode                             
bb_only           9 -1.623 -1.890  -1.947
bb_stoch         21 -1.394 -2.011  -2.052



## Heuristic G — calendar-year regime robustness for each group’s top 3

Use post-microstructure equity and the package Sharpe implementation. The comparison window is 2019–2025: partial 2019 is retained, while partial 2026 is excluded. The soft condition is at least three positive bins.

In [7]:
regime_robustness = {}
for key, group in df.groupby(GROUP, sort=True):
    label = "|".join(key)
    regime_robustness[label] = []
    for rank, row in enumerate(group.sort_values("sharpe_raw", ascending=False).head(3).itertuples(index=False), start=1):
        equity_path = ROOT / "results" / "backtests" / row.run_id / "equity.csv"
        eq = pd.read_csv(equity_path, parse_dates=["datetime"]).set_index("datetime")["Equity_adjusted"].sort_index()
        yearly = {}
        for year, series in eq.loc[eq.index.year <= 2025].groupby(eq.loc[eq.index.year <= 2025].index.year):
            yearly[str(year)] = sharpe(series) if len(series) > 2 else np.nan
        n_positive = sum(np.isfinite(v) and v > 0 for v in yearly.values())
        regime_robustness[label].append({
            "rank_raw": rank, "run_id": row.run_id, "full_sharpe_raw": row.sharpe_raw,
            "full_sharpe_post_reported": row.sharpe_post_reported,
            "full_sharpe_post_valid": row.sharpe_post_valid,
            "post_equity_min": row.post_equity_min, "post_equity_valid": row.post_equity_valid,
            "hard_filter_pass": bool(row.run_id in set(eligible["run_id"])),
            "yearly_sharpe_post": yearly, "n_positive_years": n_positive,
            "soft_gate_pass": bool(row.post_equity_valid and n_positive >= 3),
        })

regime_table = pd.DataFrame([
    {"group": group, **{k: v for k, v in item.items() if k != "yearly_sharpe_post"}, **item["yearly_sharpe_post"]}
    for group, items in regime_robustness.items() for item in items
])
regime_table.round(3)

log_returns: equity <= 0 (bankructwo) → pusta seria


sharpe: pusta seria zwrotow → NaN


log_returns: equity <= 0 (bankructwo) → pusta seria


sharpe: pusta seria zwrotow → NaN


log_returns: equity <= 0 (bankructwo) → pusta seria


...

log_returns: equity <= 0 (bankructwo) → pusta seria


sharpe: pusta seria zwrotow → NaN


log_returns: equity <= 0 (bankructwo) → pusta seria


sharpe: pusta seria zwrotow → NaN


,group,rank_raw,run_id,full_sharpe_raw,full_sharpe_post_reported,full_sharpe_post_valid,post_equity_min,post_equity_valid,hard_filter_pass,n_positive_years,soft_gate_pass,2019,2020,2021,2022,2023,2024,2025
0,mr_b1.yaml|BTC/USDT|1h,1,20260712_183518_mean_reversion_bb_stoch_BTCUSD...,-0.291,-0.497,-0.497,156534.934,True,False,2,False,-1.757,-1.199,0.596,-0.192,-0.897,0.059,-0.885
1,mr_b1.yaml|BTC/USDT|1h,2,20260712_183338_mean_reversion_bb_stoch_BTCUSD...,-0.347,-0.700,-0.700,50138.294,True,False,1,False,-1.253,-0.896,0.482,-0.306,-0.112,-0.595,-1.095
2,mr_b1.yaml|BTC/USDT|1h,3,20260712_182949_mean_reversion_bb_stoch_BTCUSD...,-0.364,-0.752,-0.752,6797.618,True,False,2,False,-1.597,-1.499,0.873,0.178,-0.404,-0.217,-1.589
3,mr_b1.yaml|ETH/USDT|1h,1,20260712_185056_mean_reversion_bb_stoch_ETHUSD...,-0.783,-0.034,NaN,-23205.132,False,False,1,False,-2.062,-1.276,-1.058,-1.601,0.855,NaN,NaN
4,mr_b1.yaml|ETH/USDT|1h,2,20260712_184651_mean_reversion_bb_stoch_ETHUSD...,-0.801,0.149,NaN,-41878.969,False,False,0,False,-2.554,-0.127,-1.635,-1.261,NaN,NaN,NaN
5,mr_b1.yaml|ETH/USDT|1h,3,20260712_185537_mean_reversion_bb_stoch_ETHUSD...,-0.830,0.058,NaN,-52025.443,False,False,0,False,-4.262,-0.232,-1.588,NaN,NaN,NaN,NaN
6,mr_b2.yaml|BTC/USDT|1h,1,20260712_190537_mean_reversion_bb_stoch_BTCUSD...,-1.062,-0.421,NaN,-71605.823,False,False,0,False,-1.266,-2.399,-0.356,-1.349,NaN,NaN,NaN
7,mr_b2.yaml|BTC/USDT|1h,2,20260712_191154_mean_reversion_bb_stoch_BTCUSD...,-1.112,-1.111,-1.111,23850.144,True,False,1,False,-3.245,-3.707,-0.544,0.060,-0.780,-0.811,NaN
8,mr_b2.yaml|BTC/USDT|1h,3,20260712_191123_mean_reversion_bb_stoch_BTCUSD...,-1.154,-0.445,NaN,-43832.600,False,False,0,False,-2.895,-2.961,-0.320,-1.405,NaN,NaN,NaN
9,mr_b2.yaml|ETH/USDT|1h,1,20260712_192706_mean_reversion_bb_stoch_ETHUSD...,-1.394,-0.551,NaN,-15401.919,False,False,0,False,-2.669,-2.866,-1.460,NaN,NaN,NaN,NaN


## Export evidence of record

In [8]:
def native(value):
    if isinstance(value, dict):
        return {str(k): native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [native(v) for v in value]
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.bool_,)): return bool(value)
    if pd.isna(value): return None
    return value

review = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "strategy": "mean_reversion_bb_stoch",
    "n_rows": len(df),
    "groups": {"|".join(k): int(v) for k, v in sizes.items()},
    "post_equity_audit": {
        "n_valid_positive_throughout": int(df["post_equity_valid"].sum()),
        "n_bankrupt_or_zero": int((~df["post_equity_valid"]).sum()),
        "reported_sharpe_warning": "sharpe_post from the original sweep is invalid when post_equity_min <= 0",
    },
    "wf_eligibility_thresholds": WF_ELIGIBILITY_THRESHOLDS,
    "summary": summary.to_dict(orient="records"),
    "top10_params": top10_params,
    "clustering_top10": clustering_top10,
    "cross_symbol_consistency": cross_symbol,
    "entry_mode_split_b2": entry_mode_split,
    "eligible_candidates": [
        {"run_id": row.run_id, "space_file": row.space_file, "symbol": row.symbol, "timeframe": row.timeframe,
         "sharpe_post": row.sharpe_post_valid, "profit_factor_post": row.profit_factor_post,
         "max_drawdown_pct_post": row.max_drawdown_pct_post, "n_trades_post": row.n_trades_post,
         "selection_score": row.selection_score, "params": json.loads(row.params)}
        for row in eligible.itertuples(index=False)
    ],
    "regime_robustness_top3_per_group": regime_robustness,
}
REVIEW_JSON.write_text(json.dumps(native(review), indent=2, sort_keys=True) + "\n")
print(REVIEW_JSON, REVIEW_JSON.stat().st_size, "bytes")

/home/janek/quant_projects/algo_bot/results/experiments/mr_sweep_review.json 82893 bytes


## Post-sweep interpretation

### Verdict: **FAILS as a standalone bare core**

The hard pre-WF gate admits **0/180** runs. The failure is not borderline: the best raw Sharpe is −0.291, every group has a negative raw top-5 mean, and only 11 post-cost curves remain positive throughout. The other **169/180 reach equity ≤ 0**. Among the 11 valid post-cost curves, the best Sharpe_post is −0.497. No current parameter set is eligible for walk-forward.

The originally stored positive Sharpe_post values are not evidence. The audit found that the pre-session `log_returns()` implementation silently discarded NaNs after equity crossed zero and could calculate Sharpe only on the pre-bankruptcy prefix. The production metric now fails closed for non-positive equity, and this review ranks only valid post-cost values (using raw Sharpe for diagnostics where no valid post-cost ranking exists).

### Prior versus evidence

1. **Confirmed, more strongly than expected:** the standalone base fails rather than landing in the marginal 0.5–1.0 band. This tests the implemented base, not the deferred MMS edge.
2. **Mostly confirmed:** b2 `bb_only` has a better raw mean and median than `bb_stoch` on both assets. BTC top-10 strongly enriches `bb_only` (6/10 despite only 9/30 samples); ETH top-10 is 3/10, equal to its 9/30 sampling share. There is no robust evidence that the Beta Stoch base-entry adaptation rescues the strategy.
3. **Not cleanly measurable as a Sharpe spread:** microstructure pushes most curves through zero, invalidating log-return Sharpe. The economic conclusion is stronger than a large penalty: the post-cost strategy is insolvent in 169 runs.
4. **Trade-count prior rejected:** all runs have far more than 100 trades (roughly 1.5k–10.3k). The failure is not small-sample noise.
5. **Consistent failure across symbols:** BTC and ETH raw top-5 means are negative for every space; no positive cross-symbol edge exists.
6. **Confirmed:** no top-3 candidate clears the soft regime gate. Even the best valid b1/BTC runs have only 1–2 positive year bins.
7. **Confirmed at the lower bound:** zero hard-filter survivors.

### MR-Session 3 decision

Do **not** spend compute on walk-forward for this bare core. The methodology-level question remains open because MMS assigns its claimed protection to pyramiding plus sequential leverage reduction, which this engine/run did not test. MR-Session 3 should therefore be a bounded **pyramiding ADR + engine migration cost/benefit** decision:

- proceeding directly to WF as a marginal candidate: **rejected** (the edge is not marginal);
- specifying the MMS state machine as a rescue hypothesis before WF: **the only methodology-consistent continuation**;
- migrating/prototyping an engine that can express add-ons and x1→x0.1 sequencing: proceed only if its cost is justified relative to pivoting to the next candidate.

If that ADR declines migration, record a no-go for the implemented MR line and pivot. A weak bare-core sweep does **not** by itself falsify the full MMS methodology.